In [ ]:
import os
import base64
import numpy as np
import plotly
import imageio.v3 as imageio
from scipy.ndimage import zoom
from IPython.display import Image
import matplotlib.pyplot as plt

from gvs_propagator.constants import nm, um, mm, cm
from gvs_propagator.coherent import (
    compute_angular_spectrum,
    compute_angular_spectrum_f
)
from gvs_propagator.partially_coherent import (
    compute_complex_coherence_factor,
    compute_angular_spectrum_gvs,
)
from gvs_propagator.utils import normalize, crop_center
from gvs_propagator.masks import create_circular_mask
from gvs_propagator.notebooks.utils import display_image

plotly.offline.init_notebook_mode()


# PARAMETERS

num_pixels_x = ## Specify number of pixels in x-direction ##
num_pixels_y = ## Specify number of pixels in y-direction ##
wavelength      = ## Specify wavelength in meters ##
z_source_object = ## Specify source to object distance in meters ##
source_radii = ## Specify source radii in meters ##
dx_camera = dy_camera =  ## Specify camera pixel pitch in meters ##
pad_factor = ## Specify padding factor ##

propagation_distances = ## Specify propagation distance ##

# OBJECT DATASET ROOT
objects_root = ## Specify path to the object dataset root directory ##
object_groups = ## [Specify object classes] ##
object_ids = ## [Specify object IDs] ##


# HELPERS
def zoom_image(image):
    return zoom(
        image,
        (
            num_pixels_y / image.shape[0],
            num_pixels_x / image.shape[1]
        )
    )

def save_true_size_png(path, img01):
    
    img01 = np.clip(img01, 0.0, 1.0)
    img8 = (img01 * 255.0 + 0.5).astype(np.uint8)
    imageio.imwrite(path, img8)


# MAIN FUNCTION

def show_diffraction():


    out_dir = ## Specify output directory for simulated measurements ##
    os.makedirs(out_dir, exist_ok=True)

    coh_names = ## Specify coherence names corresponding to source_radii ##

    # We will store only the three simulated GVS images for display as a triplet
    sim_triplet = []

    # LOOP OVER ALL 16 OBJECTS
    for obj_group in object_groups:
        for obj_id in object_ids:

            obj_path = os.path.join(objects_root, obj_group, f"{obj_id}.png")

            # --- load & normalize the amplitude-only object ---
            amp = normalize(
                zoom_image(
                    imageio.imread(
                        obj_path,
                        mode="L"
                    )
                )
            )


            # --- build amplitude-only complex object (zero phase) ---
            obj_complex = amp.astype(np.complex64)   # same as amp * exp(1j*0)

            # --- zero-pad by one full object size on each side to avoid FFT aliasing ---
            pad_y = pad_factor * num_pixels_y
            pad_x = pad_factor * num_pixels_x
            obj_padded = np.pad(
                obj_complex,
                ((pad_y, pad_y), (pad_x, pad_x)),
                mode="constant",
                constant_values=1
            )
            zpad = obj_padded.shape[0]  # FFT grid size

            for z in propagation_distances:
                # compute sampling pitches on object/source planes
                ny, nx = obj_padded.shape
                dx_obj = (wavelength * z) / (2 * dx_camera * nx)
                dy_obj = (wavelength * z) / (2 * dy_camera * ny)
                dx_src = (wavelength * z_source_object) / (2 * dx_obj * nx)
                dy_src = dx_src

                # ---- PARTIALLY COHERENT ----
                sim_triplet = []  # reset for this z (only one z anyway)

                for k, r in enumerate(source_radii):
                    src_I = create_circular_mask(nx, dx_src, r, ny, dy_src)
                    gamma = compute_complex_coherence_factor(src_I)

                    Igvs = normalize(crop_center(
                        compute_angular_spectrum_gvs(
                            obj_padded, gamma,
                            z_source_object, z,
                            dx_camera, wavelength
                        ),
                        num_pixels_y, num_pixels_x
                    ))

                    sim_triplet.append(Igvs)

                    # save simulated measurements
                    subdir = os.path.join(out_dir, obj_group, obj_id, coh_names[k])
                    os.makedirs(subdir, exist_ok=True)
                    save_true_size_png(os.path.join(subdir, "Simulated_2mm.png"), Igvs)

    print(f"Saved simulated measurements to: {os.path.abspath(out_dir)}")

    # DISPLAY SIMULATED MEASUREMENTS
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=150)
    for ax, im, title in zip(axes, sim_triplet, coh_names):
        ax.imshow(im, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title.replace("_", " "), fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# Run
show_diffraction()